In [1]:
# import logging
# logging.basicConfig(level=logging.DEBUG)

from sentence_transformers import SentenceTransformer


In [2]:

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.323324)

In [5]:
from ingest import load_faq_data

documents = load_faq_data()

In [6]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [7]:
len(texts)

1349

In [8]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1349

In [9]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

scores

[np.float32(0.4874059),
 np.float32(0.20991936),
 np.float32(0.762941),
 np.float32(0.44378537),
 np.float32(0.26083994),
 np.float32(0.4866516),
 np.float32(0.3006156),
 np.float32(0.5600999),
 np.float32(0.45960486),
 np.float32(0.25628167),
 np.float32(0.33153275),
 np.float32(0.2731852),
 np.float32(0.2769164),
 np.float32(0.34122992),
 np.float32(0.46007162),
 np.float32(0.26240283),
 np.float32(0.39200068),
 np.float32(0.10854162),
 np.float32(0.27567297),
 np.float32(0.16646823),
 np.float32(0.31437916),
 np.float32(0.0066852663),
 np.float32(0.1120502),
 np.float32(0.21905696),
 np.float32(0.3400085),
 np.float32(0.23571283),
 np.float32(0.32714835),
 np.float32(0.1508836),
 np.float32(0.16563274),
 np.float32(0.055450246),
 np.float32(0.23541188),
 np.float32(0.085330114),
 np.float32(-0.0030899458),
 np.float32(-0.042598397),
 np.float32(-0.060277134),
 np.float32(0.006491117),
 np.float32(0.034277458),
 np.float32(-0.049589735),
 np.float32(-0.0006708391),
 np.float32(-0.017

In [10]:
import numpy as np

X = np.array(vectors)
scores = X.dot(v1)


In [11]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [12]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [13]:
top5 = np.argsort(-scores)[0:5]
print(top5)
scores[top5]

[  2 624 906 538   7]


array([0.762941  , 0.7579372 , 0.71921325, 0.6536312 , 0.56009996],
      dtype=float32)

In [14]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.71921325
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions',

In [15]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [16]:
vindex.search(v1, 
              num_results=5, 
              filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [17]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [18]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [19]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [20]:
query = "Ollama"
assistant.rag(query)

'To install Ollama:\n\n- Go to: https://ollama.com/download\n- Choose your OS:\n  - **macOS**: download the `.pkg` installer\n  - **Windows**: download the `.msi` installer\n  - **Linux**: run:\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\nAfter installing, test it with:\n\n```bash\nollama run llama3\n```\n\nTo check the local server:\n\n```bash\ncurl http://localhost:11434\n```\n\nThen install the Python client:\n\n```bash\npip install ollama\n```\n\nIf you want, I can also give you the minimal Python example.'

In [21]:
from rag_helper import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [22]:
vector_assistant.rag("Ollama")

'Install Ollama from: https://ollama.com/download\n\n- **macOS**: download the `.pkg` and install it\n- **Windows**: download the `.msi` and install it\n- **Linux**: run:\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\nThen start it with:\n\n```bash\nollama run llama3\n```\n\nTo check the local server:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you want, I can also show you the Python client example.'

In [23]:
from sqlitesearch import VectorSearchIndex


In [24]:

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [25]:
vs_index.fit(vectors,documents)

In [28]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)


In [29]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [30]:
vs_index.close()